# Diabetes Health Indicators - Exploratory Data Analysis (BRFSS 2015) AI4ALL Group 23E

This repository contains an exploratory data analysis focused on identifying relationships between various health indicators, demographic factors, and diabetes status using the 2015 Behavioral Risk Factor Surveillance System (BRFSS) dataset. The visualizations highlight the interactions between key predictors and our target variable: `Diabetes_binary`.

## Key Insights & Visualizations

To avoid data imbalance distortions, a balanced 50-50 split dataset (`diabetes_binary_5050split_health_indicators_BRFSS2015.csv`) was utilized for the following analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the 50-50 split dataset
df = pd.read_csv('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
sns.set_theme(style="whitegrid")

### 1. Global Variable Correlations
The correlation matrix heatmap provides an overarching look at how variables move together.
* **Top Positive Predictors:** General Health (`GenHlth`), High Blood Pressure (`HighBP`), Body Mass Index (`BMI`), and High Cholesterol (`HighChol`) share the strongest linear relationships with diabetes status.
* **Secondary Trait Clusters:** Strong correlations also appear between physical health issues (`PhysHlth`) and difficulty walking (`DiffWalk`).

In [ ]:
plt.figure(figsize=(14, 11))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm")
plt.tight_layout()
plt.savefig('correlation_matrix_heatmap.png')
plt.show()

### 2. Analysis of Primary Predictors
Focusing on the highest-correlated indicators, the bar charts below show the proportion of individuals diagnosed with diabetes across different categories. **Bars are strictly sorted by descending prevalence** to emphasize risk tiers:

* **General Health Rating:** Individuals reporting "Poor" (5) or "Fair" (4) health exhibit a drastically higher rate of diabetes compared to those reporting "Excellent" (1) health.
* **Comorbidities:** Having high blood pressure or high cholesterol nearly doubles the statistical prevalence of diabetes within the sample population.

In [ ]:
predictors = ['GenHlth', 'HighBP', 'HighChol', 'DiffWalk']

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.ravel(), predictors):
    rates = df.groupby(col)['Diabetes_binary'].mean().sort_values(ascending=False)
    sns.barplot(x=rates.index.astype(str), y=rates.values, ax=ax, color='#4C72B0')
    ax.set_title(f'Diabetes Rate by {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Diabetes Rate')

plt.tight_layout()
plt.savefig('top_predictors_bar_charts.png')
plt.show()

### 3. The Compounding Risk of BMI and Age
Because `Age` (1–13 bracket scale) and `BMI` are discrete/dense values, a standard scatter plot causes heavy overplotting. By extracting a random sample of 3,000 individuals and applying a subtle coordinate **jitter**, a clear risk transition zone emerges:
* **The Trend:** Moving toward the upper-right quadrant (higher BMI + advanced age bracket) shows a severe, undeniable concentration of individuals with diabetes (represented in Red).

In [ ]:
sample = df.sample(3000, random_state=42).copy()

rng = np.random.default_rng(42)
sample['Age_jitter'] = sample['Age'] + rng.uniform(-0.3, 0.3, size=len(sample))
sample['BMI_jitter'] = sample['BMI'] + rng.uniform(-0.6, 0.6, size=len(sample))

colors = sample['Diabetes_binary'].map({0: '#4C72B0', 1: '#C44E52'})

plt.figure(figsize=(10, 7))
plt.scatter(sample['Age_jitter'], sample['BMI_jitter'], c=colors, alpha=0.5, s=20)
plt.xlabel('Age Group (1-13 bracket, jittered)')
plt.ylabel('BMI (jittered)')
plt.title('BMI vs. Age Group, Colored by Diabetes Status (Red = Diabetic)')
plt.tight_layout()
plt.savefig('bmi_vs_age_scatter_jitter.png')
plt.show()

### 4. Socioeconomic Protective Factors
Plotting protective socioeconomic factors reveals an inverse trend with diabetes status:
* **The Trend:** As both **Income Level** (1–8 scale) and **Education Level** (1–6 scale) increase, the proportion of diabetes diagnoses steadily and cleanly drops off, highlighting a strong negative correlation between socioeconomic status and diabetes risk.

In [ ]:
income_rate = df.groupby('Income')['Diabetes_binary'].mean()
education_rate = df.groupby('Education')['Diabetes_binary'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(income_rate.index, income_rate.values, marker='o', color='#55A868')
axes[0].set_title('Diabetes Rate by Income Level')
axes[0].set_xlabel('Income Level (1-8)')
axes[0].set_ylabel('Diabetes Rate')

axes[1].plot(education_rate.index, education_rate.values, marker='o', color='#8172B2')
axes[1].set_title('Diabetes Rate by Education Level')
axes[1].set_xlabel('Education Level (1-6)')
axes[1].set_ylabel('Diabetes Rate')

plt.tight_layout()
plt.savefig('socioeconomic_vs_diabetes_line.png')
plt.show()